# IPAM R2 — Final Two-Dataset Google Colab Pipeline

This notebook is designed specifically for the two supplied CSV files.

### Dataset A — five-class
`Sarcasm_multiclassification_1(1).csv`

Expected:
- 3,460 rows
- `comment_Text`
- `Have_Sarcasm`
- `Sarcasm_Type`
- five sarcasm classes:
  - Humor
  - Irony
  - Flattery
  - Self-Deprecation
  - Passive-Aggression

### Dataset B — binary
`annotated_comments_binary_1(1).csv`

Expected:
- 9,999 rows
- `comment_Text`
- `Have_Sarcasm`
- `Sarcasm_Type`
- `Have_Sarcasm`:
  - No = Non-Sarcastic
  - Yes = Sarcastic

## Important

Do **not** concatenate the two files. They are separate experimental sources/tasks.

The five-class experiment uses Dataset A.

The binary experiment uses Dataset B.

RandomOverSampler is applied **only to the training partition** and only after checking that the target has at least two classes.

The `Without FA` / `With FA` labels are retained only as manuscript/display labels. Both conditions in this notebook use **standard Transformer attention**. No FlashAttention package or CUDA kernel is used.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd

OUTPUT_DIR = "/content/drive/MyDrive/IPAM_R21_FINAL_RESULTS"

metrics_dir = os.path.join(
    OUTPUT_DIR,
    "metrics"
)

print(os.listdir(metrics_dir))

['binary_BERT_seed42_per_class.csv', 'binary_RoBERTa_seed42_per_class.csv', 'binary_DistilBERT_seed42_per_class.csv', 'binary_TinyBERT_seed42_per_class.csv', 'binary_MiniLM_seed42_per_class.csv', 'binary_BERT_seed123_per_class.csv', 'binary_RoBERTa_seed123_per_class.csv', 'binary_DistilBERT_seed123_per_class.csv', 'binary_TinyBERT_seed123_per_class.csv', 'binary_MiniLM_seed123_per_class.csv', 'all_results_progress.csv']


In [ ]:
progress_file = os.path.join(
    OUTPUT_DIR,
    "metrics",
    "all_results_progress.csv"
)

final_file = os.path.join(
    OUTPUT_DIR,
    "metrics",
    "all_encoder_results.csv"
)

if os.path.exists(progress_file):
    existing = pd.read_csv(progress_file)
elif os.path.exists(final_file):
    existing = pd.read_csv(final_file)
else:
    existing = pd.DataFrame()

print("Existing completed runs:", len(existing))

if len(existing) > 0:
    display(
        existing[
            [
                "task",
                "model",
                "seed",
                "Macro_F1"
            ]
        ].sort_values(
            ["task", "model", "seed"]
        )
    )

Existing completed runs: 10


,task,model,seed,Macro_F1
0,binary,BERT,42,0.992608
5,binary,BERT,123,0.993356
2,binary,DistilBERT,42,0.993350
7,binary,DistilBERT,123,0.994832
4,binary,MiniLM,42,0.990385
9,binary,MiniLM,123,0.991880
1,binary,RoBERTa,42,0.986694
6,binary,RoBERTa,123,0.990412
3,binary,TinyBERT,42,0.976344
8,binary,TinyBERT,123,0.980833


In [ ]:
# ============================================================
# 0. INSTALL REQUIRED PACKAGES
# ============================================================
# Run this cell once in a fresh Colab runtime.
# We intentionally do NOT install flash-attn.

!pip -q install -U transformers accelerate imbalanced-learn scipy scikit-learn statsmodels sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 99.0 MB/s eta 0:00:00


In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import os
import re
import gc
import json
import time
import math
import random
import platform
import warnings
from datetime import datetime
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    f1_score,
)

from imblearn.over_sampling import RandomOverSampler
from scipy.stats import t as student_t
from statsmodels.stats.contingency_tables import mcnemar

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", __import__("transformers").__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Python: 3.13.15
PyTorch: 2.11.0+cu128
Transformers: 5.16.1
CUDA available: True
GPU: Tesla T4
CUDA: 12.8
Device: cuda


In [ ]:
# ============================================================
# 2. GOOGLE DRIVE + DATASET LOCATION
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

# ------------------------------------------------------------
# Option 1: Put both CSV files in this Drive folder.
# ------------------------------------------------------------

DATA_DIR = "/content/drive/MyDrive/IPAM_R2_DATA"

FIVE_CLASS_FILENAME = "Sarcasm_multiclassification_1.csv"
BINARY_FILENAME = "annotated_comments_binary_1.csv"

FIVE_CLASS_PATH = os.path.join(DATA_DIR, FIVE_CLASS_FILENAME)
BINARY_PATH = os.path.join(DATA_DIR, BINARY_FILENAME)

# ------------------------------------------------------------
# If your filenames/locations are different, change the two
# paths above.
# ------------------------------------------------------------

OUTPUT_DIR = "/content/drive/MyDrive/IPAM_R21_FINAL_RESULTS"

for d in [
    "environment",
    "data_audit",
    "splits",
    "predictions",
    "metrics",
    "confusion_matrices",
    "timing",
    "statistics",
    "qualitative",
    "mistral",
    "fatima",
    "manuscript",
    "audit",
]:
    os.makedirs(os.path.join(OUTPUT_DIR, d), exist_ok=True)

print("Five-class dataset:", FIVE_CLASS_PATH)
print("Binary dataset:", BINARY_PATH)
print("Output directory:", OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Five-class dataset: /content/drive/MyDrive/IPAM_R2_DATA/Sarcasm_multiclassification_1.csv
Binary dataset: /content/drive/MyDrive/IPAM_R2_DATA/annotated_comments_binary_1.csv
Output directory: /content/drive/MyDrive/IPAM_R21_FINAL_RESULTS


## If the files are not in Google Drive

If the next cell says a file is missing, upload the CSV files to:

`/content/drive/MyDrive/IPAM_R2_DATA/`

with these exact names:

- `Sarcasm_multiclassification_1(1).csv`
- `annotated_comments_binary_1(1).csv`

Do not rename the files unless you also change the configuration cell.

In [ ]:
# ============================================================
# 3. LOAD BOTH DATASETS
# ============================================================

if not os.path.exists(FIVE_CLASS_PATH) or not os.path.exists(BINARY_PATH):
    from google.colab import files

    print("One or both Drive files were not found.")
    print("Please upload BOTH CSV files now.")
    uploaded = files.upload()

    # Locate uploaded files by their expected names.
    for name in uploaded.keys():
        if name == FIVE_CLASS_FILENAME:
            FIVE_CLASS_PATH = os.path.join("/content", name)
        elif name == BINARY_FILENAME:
            BINARY_PATH = os.path.join("/content", name)

if not os.path.exists(FIVE_CLASS_PATH):
    raise FileNotFoundError(
        "Five-class CSV not found. Expected:\n" + FIVE_CLASS_PATH
    )

if not os.path.exists(BINARY_PATH):
    raise FileNotFoundError(
        "Binary CSV not found. Expected:\n" + BINARY_PATH
    )

five_raw = pd.read_csv(FIVE_CLASS_PATH, encoding="latin1")
binary_raw = pd.read_csv(BINARY_PATH, encoding="latin1")

print("Five-class dataset shape:", five_raw.shape)
print("Five-class columns:", five_raw.columns.tolist())

print("\nBinary dataset shape:", binary_raw.shape)
print("Binary columns:", binary_raw.columns.tolist())

Five-class dataset shape: (3460, 4)
Five-class columns: ['comment_Text', 'Have_Sarcasm', 'Sarcasm_Type', 'Unnamed: 3']

Binary dataset shape: (9999, 3)
Binary columns: ['comment_Text', 'Have_Sarcasm', 'Sarcasm_Type']


In [ ]:
# ============================================================
# 4. LABEL NORMALIZATION
# ============================================================

REQUIRED_TEXT_COLUMN = "comment_Text"
REQUIRED_SARCASM_COLUMN = "Sarcasm_Type"
REQUIRED_BINARY_COLUMN = "Have_Sarcasm"

def validate_columns(df, required, name):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"{name} is missing required columns: {missing}\n"
            f"Available columns: {df.columns.tolist()}"
        )

validate_columns(
    five_raw,
    [REQUIRED_TEXT_COLUMN, REQUIRED_SARCASM_COLUMN],
    "Five-class dataset"
)

validate_columns(
    binary_raw,
    [REQUIRED_TEXT_COLUMN, REQUIRED_BINARY_COLUMN],
    "Binary dataset"
)

def clean_text(x):
    return re.sub(r"\s+", " ", str(x)).strip()

def normalize_sarcasm_type(x):
    if pd.isna(x):
        return np.nan

    x = re.sub(r"\s+", " ", str(x).strip())

    aliases = {
        "Passive-Aggression": "Passive-Aggression",
        "Passive Aggression": "Passive-Aggression",

        "Self-Deprecation": "Self-Deprecation",
        "Self Deprecation": "Self-Deprecation",

        "Humor": "Humor",
        "Irony": "Irony",
        "Flattery": "Flattery",
    }

    return aliases.get(x, x)

def normalize_have_sarcasm(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip().lower()

    if x in {"yes", "y", "1", "1.0", "true", "sarcastic"}:
        return "Sarcastic"

    if x in {"no", "n", "0", "0.0", "false", "non-sarcastic"}:
        return "Non-Sarcastic"

    return np.nan


five_df = five_raw.copy()
binary_df = binary_raw.copy()

five_df["original_row_id"] = np.arange(len(five_df))
binary_df["original_row_id"] = np.arange(len(binary_df))

five_df["comment_Text"] = five_df["comment_Text"].fillna("").astype(str)
binary_df["comment_Text"] = binary_df["comment_Text"].fillna("").astype(str)

five_df["text_clean"] = five_df["comment_Text"].apply(clean_text)
binary_df["text_clean"] = binary_df["comment_Text"].apply(clean_text)

five_df["Sarcasm_Type"] = five_df["Sarcasm_Type"].apply(
    normalize_sarcasm_type
)

binary_df["Sarcasm_Type"] = binary_df["Sarcasm_Type"].apply(
    normalize_sarcasm_type
)

binary_df["Have_Sarcasm_Normalized"] = binary_df["Have_Sarcasm"].apply(
    normalize_have_sarcasm
)

print("Normalization completed.")

Normalization completed.


In [ ]:
# ============================================================
# 5. DATASET VALIDATION — HARD CHECKS
# ============================================================

FIVE_CLASSES = [
    "Humor",
    "Irony",
    "Flattery",
    "Self-Deprecation",
    "Passive-Aggression",
]

BINARY_CLASSES = [
    "Non-Sarcastic",
    "Sarcastic",
]

# -----------------------------
# Five-class dataset
# -----------------------------
five_class_df = five_df[
    five_df["Sarcasm_Type"].isin(FIVE_CLASSES)
].copy()

unknown_five = sorted(
    set(five_df["Sarcasm_Type"].dropna().unique()) - set(FIVE_CLASSES)
)

if unknown_five:
    print("WARNING: Unknown five-class labels:", unknown_five)

missing_five = [
    c for c in FIVE_CLASSES
    if c not in set(five_class_df["Sarcasm_Type"])
]

if missing_five:
    raise ValueError(
        f"Five-class dataset is missing classes: {missing_five}"
    )

five_class_df["label_name"] = five_class_df["Sarcasm_Type"]

# -----------------------------
# Binary dataset
# -----------------------------
binary_valid = binary_df[
    binary_df["Have_Sarcasm_Normalized"].isin(BINARY_CLASSES)
].copy()

if binary_valid["Have_Sarcasm_Normalized"].isna().any():
    raise ValueError(
        "Binary dataset contains invalid/unknown Have_Sarcasm values."
    )

binary_valid["label_name"] = binary_valid[
    "Have_Sarcasm_Normalized"
]

if binary_valid["label_name"].nunique() != 2:
    raise ValueError(
        "Binary dataset does not contain exactly two classes."
    )

# -----------------------------
# Remove empty comments
# -----------------------------
five_class_df = five_class_df[
    five_class_df["text_clean"].str.len() > 0
].reset_index(drop=True)

binary_valid = binary_valid[
    binary_valid["text_clean"].str.len() > 0
].reset_index(drop=True)

print("=" * 70)
print("FIVE-CLASS DATASET")
print("=" * 70)
print("Rows:", len(five_class_df))
print(five_class_df["label_name"].value_counts())
print()

print("=" * 70)
print("BINARY DATASET")
print("=" * 70)
print("Rows:", len(binary_valid))
print(binary_valid["label_name"].value_counts())

FIVE-CLASS DATASET
Rows: 3460
label_name
Humor                 1709
Irony                  536
Flattery               513
Self-Deprecation       439
Passive-Aggression     263
Name: count, dtype: int64

BINARY DATASET
Rows: 9999
label_name
Non-Sarcastic    6539
Sarcastic        3460
Name: count, dtype: int64


In [ ]:
# ============================================================
# 6. EXPECTED-DATASET AUDIT
# ============================================================

# These are the counts found in the actual supplied files.
# The code reports differences rather than silently changing data.

EXPECTED_FIVE_COUNTS = {
    "Humor": 1709,
    "Irony": 536,
    "Flattery": 513,
    "Self-Deprecation": 439,
    "Passive-Aggression": 263,
}

EXPECTED_BINARY_COUNTS = {
    "Non-Sarcastic": 6539,
    "Sarcastic": 3460,
}

actual_five_counts = five_class_df["label_name"].value_counts().to_dict()
actual_binary_counts = binary_valid["label_name"].value_counts().to_dict()

print("Expected five-class counts:")
print(EXPECTED_FIVE_COUNTS)
print("\nActual five-class counts:")
print(actual_five_counts)

print("\nExpected binary counts:")
print(EXPECTED_BINARY_COUNTS)
print("\nActual binary counts:")
print(actual_binary_counts)

if actual_five_counts != EXPECTED_FIVE_COUNTS:
    raise ValueError(
        "Five-class counts differ from the supplied dataset counts. "
        "Inspect the CSV before running experiments."
    )

if actual_binary_counts != EXPECTED_BINARY_COUNTS:
    raise ValueError(
        "Binary counts differ from the supplied dataset counts. "
        "Inspect the CSV before running experiments."
    )

print("\nDATASET COUNT VALIDATION: PASSED")

Expected five-class counts:
{'Humor': 1709, 'Irony': 536, 'Flattery': 513, 'Self-Deprecation': 439, 'Passive-Aggression': 263}

Actual five-class counts:
{'Humor': 1709, 'Irony': 536, 'Flattery': 513, 'Self-Deprecation': 439, 'Passive-Aggression': 263}

Expected binary counts:
{'Non-Sarcastic': 6539, 'Sarcastic': 3460}

Actual binary counts:
{'Non-Sarcastic': 6539, 'Sarcastic': 3460}

DATASET COUNT VALIDATION: PASSED


In [ ]:
# ============================================================
# 7. DUPLICATE AUDIT — WITHIN EACH DATASET
# ============================================================

def duplicate_report(df, dataset_name):
    duplicated = df["text_clean"].duplicated(keep=False)

    print("\n" + "=" * 70)
    print(dataset_name)
    print("=" * 70)
    print("Rows:", len(df))
    print("Unique texts:", df["text_clean"].nunique())
    print("Rows belonging to duplicated-text groups:", int(duplicated.sum()))
    print("Duplicate groups:", int(
        df.loc[duplicated, "text_clean"].nunique()
    ))

    return {
        "dataset": dataset_name,
        "rows": len(df),
        "unique_texts": int(df["text_clean"].nunique()),
        "duplicated_rows": int(duplicated.sum()),
        "duplicate_groups": int(df.loc[duplicated, "text_clean"].nunique()),
    }

duplicate_audit = pd.DataFrame([
    duplicate_report(five_class_df, "five_class"),
    duplicate_report(binary_valid, "binary"),
])

duplicate_audit.to_csv(
    os.path.join(
        OUTPUT_DIR, "data_audit",
        "within_dataset_duplicate_audit.csv"
    ),
    index=False
)

display(duplicate_audit)


five_class
Rows: 3460
Unique texts: 3216
Rows belonging to duplicated-text groups: 487
Duplicate groups: 243

binary
Rows: 9999
Unique texts: 9948
Rows belonging to duplicated-text groups: 82
Duplicate groups: 31


,dataset,rows,unique_texts,duplicated_rows,duplicate_groups
0,five_class,3460,3216,487,243
1,binary,9999,9948,82,31


In [ ]:
# ============================================================
# 8. CROSS-DATASET OVERLAP AUDIT
# ============================================================
# The two files should NOT be concatenated.
# We only measure overlap for transparency.

five_texts = set(five_class_df["text_clean"])
binary_texts = set(binary_valid["text_clean"])

overlap = five_texts & binary_texts

print("Five-class unique texts:", len(five_texts))
print("Binary unique texts:", len(binary_texts))
print("Cross-dataset text overlap:", len(overlap))

cross_audit = pd.DataFrame([{
    "five_class_unique_texts": len(five_texts),
    "binary_unique_texts": len(binary_texts),
    "cross_dataset_overlap": len(overlap),
    "datasets_concatenated": False,
}])

cross_audit.to_csv(
    os.path.join(
        OUTPUT_DIR, "data_audit",
        "cross_dataset_overlap_audit.csv"
    ),
    index=False
)

display(cross_audit)

Five-class unique texts: 3216
Binary unique texts: 9948
Cross-dataset text overlap: 2505


,five_class_unique_texts,binary_unique_texts,cross_dataset_overlap,datasets_concatenated
0,3216,9948,2505,False


## 9. Experimental configuration

The five-class task uses the 3,460-row multiclass CSV.

The binary task uses the 9,999-row binary CSV.

Each task is split independently using stratification.

Oversampling is performed only on the training split.

In [ ]:
# ============================================================
# 9. CONFIGURATION
# ============================================================

SEEDS = [42, 123, 2024]

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

MAX_LEN = 128
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
GRADIENT_CLIP_NORM = 1.0

BATCH_SIZES = {
    "BERT": 16,
    "RoBERTa": 8,
    "DistilBERT": 16,
    "TinyBERT": 32,
    "MiniLM": 16,
}

MODELS = {
    "BERT": "bert-base-uncased",
    "RoBERTa": "roberta-base",
    "DistilBERT": "distilbert-base-uncased",
    "TinyBERT": "huawei-noah/TinyBERT_General_4L_312D",
    "MiniLM": "microsoft/MiniLM-L12-H384-uncased",
}

NUM_WARMUP_BATCHES = 10
NUM_TIMING_RUNS = 30
BOOTSTRAP_ITERATIONS = 5000

DISPLAY_WITHOUT_FA = "Without FA"
DISPLAY_WITH_FA = "With FA"
ACTUAL_ATTENTION_MECHANISM = "standard attention"

# Optional zero-shot Mistral baseline.
RUN_MISTRAL = False
MISTRAL_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
MISTRAL_MAX_NEW_TOKENS = 8
MISTRAL_4BIT = True

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

TASKS = {
    "binary": {
        "data": binary_valid,
        "classes": BINARY_CLASSES,
        "source": "annotated_comments_binary_1(1).csv",
    },
    "five_class": {
        "data": five_class_df,
        "classes": FIVE_CLASSES,
        "source": "Sarcasm_multiclassification_1(1).csv",
    },
}

print("Tasks:", list(TASKS.keys()))

Tasks: ['binary', 'five_class']


In [ ]:
# ============================================================
# 10. REPRODUCIBILITY + GPU UTILITIES
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Do not force deterministic GPU kernels because timing experiments
    # should represent normal GPU execution.
    torch.use_deterministic_algorithms(False)


def clear_gpu():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


environment = {
    "timestamp_utc": datetime.utcnow().isoformat(),
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "transformers": __import__("transformers").__version__,
    "cuda": str(torch.version.cuda),
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "CPU"
    ),
    "actual_attention": ACTUAL_ATTENTION_MECHANISM,
    "flash_attention_used": False,
    "display_without_fa": DISPLAY_WITHOUT_FA,
    "display_with_fa": DISPLAY_WITH_FA,
}

with open(
    os.path.join(
        OUTPUT_DIR, "environment",
        "environment.json"
    ),
    "w"
) as f:
    json.dump(environment, f, indent=2)

print(json.dumps(environment, indent=2))

{
  "timestamp_utc": "2026-08-26T16:40:20.573633",
  "python": "3.13.15",
  "pytorch": "2.11.0+cu128",
  "transformers": "5.16.1",
  "cuda": "12.8",
  "gpu": "Tesla T4",
  "actual_attention": "standard attention",
  "flash_attention_used": false,
  "display_without_fa": "Without FA",
  "display_with_fa": "With FA"
}


In [ ]:
# ============================================================
# 11. STRATIFIED SPLIT + SAFE OVERSAMPLING
# ============================================================

def make_splits(task_df, seed):
    counts = task_df["label_name"].value_counts()

    if task_df["label_name"].nunique() < 2:
        raise ValueError(
            f"Task has only one class: {counts.to_dict()}"
        )

    if counts.min() < 2:
        raise ValueError(
            f"A class has fewer than 2 examples: {counts.to_dict()}"
        )

    train_df, temp_df = train_test_split(
        task_df,
        test_size=VAL_RATIO + TEST_RATIO,
        stratify=task_df["label_name"],
        random_state=seed,
    )

    validation_fraction_of_temp = (
        VAL_RATIO / (VAL_RATIO + TEST_RATIO)
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=1 - validation_fraction_of_temp,
        stratify=temp_df["label_name"],
        random_state=seed,
    )

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


def oversample_training_only(train_df, seed):
    y = train_df["label_name"].astype(str)

    # The exact fix for the original error.
    if y.nunique() < 2:
        raise ValueError(
            "Oversampling cannot run because the training target "
            f"contains only one class: {y.unique().tolist()}"
        )

    X = train_df[
        [
            "original_row_id",
            "comment_Text",
            "text_clean",
            "label_name",
        ]
    ].copy()

    ros = RandomOverSampler(random_state=seed)

    X_resampled, y_resampled = ros.fit_resample(X, y)

    result = X_resampled.copy()
    result["label_name"] = np.asarray(y_resampled)

    return result.reset_index(drop=True)


split_records = []

for task_name, task_info in TASKS.items():
    task_df = task_info["data"]

    for seed in SEEDS:
        train_original, val_df, test_df = make_splits(
            task_df, seed
        )

        train_os = oversample_training_only(
            train_original, seed
        )

        base = f"{task_name}_seed{seed}"

        train_original.to_csv(
            os.path.join(
                OUTPUT_DIR, "splits",
                f"{base}_train_original.csv"
            ),
            index=False
        )

        train_os.to_csv(
            os.path.join(
                OUTPUT_DIR, "splits",
                f"{base}_train_oversampled.csv"
            ),
            index=False
        )

        val_df.to_csv(
            os.path.join(
                OUTPUT_DIR, "splits",
                f"{base}_validation.csv"
            ),
            index=False
        )

        test_df.to_csv(
            os.path.join(
                OUTPUT_DIR, "splits",
                f"{base}_test.csv"
            ),
            index=False
        )

        split_records.append({
            "task": task_name,
            "seed": seed,
            "train_original": len(train_original),
            "train_oversampled": len(train_os),
            "validation": len(val_df),
            "test": len(test_df),
            "train_classes_before": train_original["label_name"].nunique(),
            "train_classes_after": train_os["label_name"].nunique(),
            "validation_oversampled": False,
            "test_oversampled": False,
        })

split_audit = pd.DataFrame(split_records)

split_audit.to_csv(
    os.path.join(
        OUTPUT_DIR, "audit",
        "split_audit.csv"
    ),
    index=False
)

display(split_audit)

In [ ]:
# ============================================================
# 12. VERIFY NO TRAIN/VAL/TEST TEXT LEAKAGE
# ============================================================

leakage_rows = []

for task_name in TASKS:
    for seed in SEEDS:
        base = f"{task_name}_seed{seed}"

        train = pd.read_csv(
            os.path.join(
                OUTPUT_DIR, "splits",
                f"{base}_train_original.csv"
            )
        )
        val = pd.read_csv(
            os.path.join(
                OUTPUT_DIR, "splits",
                f"{base}_validation.csv"
            )
        )
        test = pd.read_csv(
            os.path.join(
                OUTPUT_DIR, "splits",
                f"{base}_test.csv"
            )
        )

        a = set(train["text_clean"])
        b = set(val["text_clean"])
        c = set(test["text_clean"])

        leakage_rows.append({
            "task": task_name,
            "seed": seed,
            "train_val_overlap": len(a & b),
            "train_test_overlap": len(a & c),
            "val_test_overlap": len(b & c),
        })

leakage_audit = pd.DataFrame(leakage_rows)

leakage_audit.to_csv(
    os.path.join(
        OUTPUT_DIR, "audit",
        "split_leakage_audit.csv"
    ),
    index=False
)

display(leakage_audit)

if leakage_audit[
    [
        "train_val_overlap",
        "train_test_overlap",
        "val_test_overlap"
    ]
].to_numpy().sum() != 0:
    print(
        "WARNING: Exact text duplicates cross split boundaries. "
        "Review the leakage audit before reporting results."
    )
else:
    print("SPLIT LEAKAGE CHECK: PASSED")

,task,seed,train_val_overlap,train_test_overlap,val_test_overlap
0,binary,42,9,9,2
1,binary,123,11,5,2
2,binary,2024,10,11,2
3,five_class,42,44,50,17
4,five_class,123,50,54,13
5,five_class,2024,39,48,8


In [ ]:
# ============================================================
# 13. DATASET CLASS + MODEL LOADING
# ============================================================

class ReviewDataset(Dataset):

    def __init__(self, dataframe, tokenizer, label2id):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.label2id = label2id

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        encoded = self.tokenizer(
            str(row["comment_Text"]),
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt",
        )

        item = {
            key: value.squeeze(0)
            for key, value in encoded.items()
        }

        item["labels"] = torch.tensor(
            self.label2id[row["label_name"]],
            dtype=torch.long,
        )

        return item


def load_model_and_tokenizer(checkpoint, class_names):

    label2id = {
        label: idx
        for idx, label in enumerate(class_names)
    }

    id2label = {
        idx: label
        for label, idx in label2id.items()
    }

    tokenizer = AutoTokenizer.from_pretrained(
        checkpoint
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=len(class_names),
        label2id=label2id,
        id2label=id2label,
        ignore_mismatched_sizes=True,
    )

    return tokenizer, model, label2id, id2label

In [ ]:
# ============================================================
# 14. EVALUATION FUNCTIONS
# ============================================================

def evaluate_model(model, loader):

    model.eval()

    all_true = []
    all_pred = []
    all_confidence = []

    with torch.no_grad():

        for batch in loader:

            labels = batch["labels"].cpu().numpy()

            device_batch = {
                key: value.to(DEVICE)
                for key, value in batch.items()
            }

            output = model(**device_batch)

            probabilities = torch.softmax(
                output.logits,
                dim=-1
            )

            predictions = (
                probabilities
                .argmax(dim=-1)
                .cpu()
                .numpy()
            )

            confidence = (
                probabilities
                .max(dim=-1)
                .values
                .cpu()
                .numpy()
            )

            all_true.extend(labels.tolist())
            all_pred.extend(predictions.tolist())
            all_confidence.extend(confidence.tolist())

    y_true = np.asarray(all_true)
    y_pred = np.asarray(all_pred)
    confidence = np.asarray(all_confidence)

    p_macro, r_macro, f_macro, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        )
    )

    p_weighted, r_weighted, f_weighted, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        )
    )

    metrics = {
        "Accuracy": accuracy_score(
            y_true, y_pred
        ),
        "Macro_Precision": p_macro,
        "Macro_Recall": r_macro,
        "Macro_F1": f_macro,
        "Weighted_Precision": p_weighted,
        "Weighted_Recall": r_weighted,
        "Weighted_F1": f_weighted,
    }

    return metrics, y_true, y_pred, confidence


def make_per_class_metrics(
    y_true,
    y_pred,
    class_names
):

    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(len(class_names))),
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )

    rows = []

    for label in class_names:

        item = report.get(label, {})

        rows.append({
            "Class": label,
            "Precision": item.get(
                "precision", np.nan
            ),
            "Recall": item.get(
                "recall", np.nan
            ),
            "F1": item.get(
                "f1-score", np.nan
            ),
            "Support": item.get(
                "support", np.nan
            ),
        })

    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# 15. TRAINING FUNCTION
# ============================================================

def train_model(
    model,
    train_loader,
    val_loader,
    seed
):

    set_seed(seed)

    model.to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    total_steps = max(
        1,
        len(train_loader) * EPOCHS
    )

    warmup_steps = int(
        total_steps * WARMUP_RATIO
    )

    scheduler = (
        get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
        )
    )

    best_val_f1 = -np.inf
    best_state = None

    start_time = time.perf_counter()

    for epoch in range(EPOCHS):

        model.train()

        epoch_loss = 0.0

        for batch in train_loader:

            batch = {
                key: value.to(DEVICE)
                for key, value in batch.items()
            }

            optimizer.zero_grad(
                set_to_none=True
            )

            output = model(**batch)

            loss = output.loss

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRADIENT_CLIP_NORM,
            )

            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()

        val_metrics, _, _, _ = evaluate_model(
            model,
            val_loader
        )

        val_f1 = val_metrics["Macro_F1"]

        if val_f1 > best_val_f1:

            best_val_f1 = val_f1

            best_state = {
                key: value.detach().cpu().clone()
                for key, value
                in model.state_dict().items()
            }

        print(
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"Loss: "
            f"{epoch_loss / max(1, len(train_loader)):.5f} | "
            f"Val Macro-F1: {val_f1:.5f}"
        )

    training_time = (
        time.perf_counter() - start_time
    )

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, training_time

In [ ]:
# ============================================================
# 16. INFERENCE LATENCY / GPU MEMORY
# ============================================================

def profile_inference(
    model,
    loader
):

    model.eval()

    batches = []

    for batch in loader:

        device_batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
            if key != "labels"
        }

        batches.append(device_batch)

        if len(batches) >= (
            NUM_WARMUP_BATCHES + NUM_TIMING_RUNS
        ):
            break

    if not batches:

        return {
            "latency_mean_ms": np.nan,
            "latency_sd_ms": np.nan,
            "latency_p95_ms": np.nan,
            "peak_inference_allocated_mb": np.nan,
            "peak_inference_reserved_mb": np.nan,
        }

    clear_gpu()

    warmup_batches = batches[
        :NUM_WARMUP_BATCHES
    ]

    timing_batches = (
        batches[NUM_WARMUP_BATCHES:]
        or batches
    )

    with torch.no_grad():

        for batch in warmup_batches:
            _ = model(**batch)

        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()

        latencies = []

        for i in range(NUM_TIMING_RUNS):

            batch = timing_batches[
                i % len(timing_batches)
            ]

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            start = time.perf_counter()

            _ = model(**batch)

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            latencies.append(
                (time.perf_counter() - start) * 1000.0
            )

    return {
        "latency_mean_ms": float(
            np.mean(latencies)
        ),
        "latency_sd_ms": float(
            np.std(latencies, ddof=1)
        ) if len(latencies) > 1 else 0.0,
        "latency_p95_ms": float(
            np.percentile(latencies, 95)
        ),
        "peak_inference_allocated_mb": (
            float(
                torch.cuda.max_memory_allocated()
                / (1024 ** 2)
            )
            if torch.cuda.is_available()
            else np.nan
        ),
        "peak_inference_reserved_mb": (
            float(
                torch.cuda.max_memory_reserved()
                / (1024 ** 2)
            )
            if torch.cuda.is_available()
            else np.nan
        ),
    }

In [ ]:
# ============================================================
# 17. RUN ONE EXPERIMENT
# ============================================================

def run_one_experiment(
    task_name,
    class_names,
    train_df,
    val_df,
    test_df,
    model_key,
    seed,
):

    checkpoint = MODELS[model_key]
    batch_size = BATCH_SIZES[model_key]

    set_seed(seed)
    clear_gpu()

    print("\n" + "=" * 78)
    print(
        f"TASK={task_name} | "
        f"MODEL={model_key} | "
        f"SEED={seed}"
    )
    print(
        f"Attention={ACTUAL_ATTENTION_MECHANISM} | "
        f"Display={DISPLAY_WITHOUT_FA}"
    )
    print("=" * 78)

    tokenizer, model, label2id, id2label = (
        load_model_and_tokenizer(
            checkpoint,
            class_names
        )
    )

    train_loader = DataLoader(
        ReviewDataset(
            train_df,
            tokenizer,
            label2id
        ),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    val_loader = DataLoader(
        ReviewDataset(
            val_df,
            tokenizer,
            label2id
        ),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    test_loader = DataLoader(
        ReviewDataset(
            test_df,
            tokenizer,
            label2id
        ),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    model, training_time = train_model(
        model,
        train_loader,
        val_loader,
        seed,
    )

    peak_training_allocated = (
        float(
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )
        if torch.cuda.is_available()
        else np.nan
    )

    peak_training_reserved = (
        float(
            torch.cuda.max_memory_reserved()
            / (1024 ** 2)
        )
        if torch.cuda.is_available()
        else np.nan
    )

    metrics, y_true, y_pred, confidence = (
        evaluate_model(
            model,
            test_loader
        )
    )

    timing = profile_inference(
        model,
        test_loader
    )

    prediction_df = test_df[
        [
            "original_row_id",
            "comment_Text",
            "text_clean",
            "label_name",
        ]
    ].copy()

    prediction_df["TrueLabel"] = [
        id2label[int(x)]
        for x in y_true
    ]

    prediction_df["Prediction"] = [
        id2label[int(x)]
        for x in y_pred
    ]

    prediction_df["Confidence"] = confidence

    prediction_df["Correct"] = (
        prediction_df["TrueLabel"]
        == prediction_df["Prediction"]
    )

    prediction_file = os.path.join(
        OUTPUT_DIR,
        "predictions",
        f"{task_name}_{model_key}_seed{seed}_WITHOUT_FA.csv"
    )

    prediction_df.to_csv(
        prediction_file,
        index=False
    )

    # Per-class metrics
    pc = make_per_class_metrics(
        y_true,
        y_pred,
        class_names
    )

    pc["task"] = task_name
    pc["model"] = model_key
    pc["seed"] = seed
    pc["display_condition"] = DISPLAY_WITHOUT_FA
    pc["actual_attention"] = ACTUAL_ATTENTION_MECHANISM

    pc.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "metrics",
            f"{task_name}_{model_key}_seed{seed}_per_class.csv"
        ),
        index=False
    )

    # Confusion matrix
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(len(class_names)))
    )

    cm_df = pd.DataFrame(
        cm,
        index=class_names,
        columns=class_names
    )

    cm_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "confusion_matrices",
            f"{task_name}_{model_key}_seed{seed}.csv"
        )
    )

    row = {
        "task": task_name,
        "model": model_key,
        "seed": seed,
        "display_condition": DISPLAY_WITHOUT_FA,
        "actual_attention": ACTUAL_ATTENTION_MECHANISM,
        "checkpoint": checkpoint,

        **metrics,

        "training_time_s": training_time,

        **timing,

        "peak_training_allocated_mb":
            peak_training_allocated,

        "peak_training_reserved_mb":
            peak_training_reserved,

        "train_original_n":
            np.nan,

        "train_oversampled_n":
            len(train_df),

        "validation_n":
            len(val_df),

        "test_n":
            len(test_df),
    }

    return row

# 18. Run all experiments

Expected number of encoder-model runs:

- Binary: 5 models × 3 seeds = **15**
- Five-class: 5 models × 3 seeds = **15**
- Total = **30 runs**

On a Tesla T4, this can take a substantial amount of time. Results are saved after every completed run, so an interrupted session does not lose previously completed CSV outputs.

In [ ]:
# ============================================================
# RESUME PREVIOUS EXPERIMENTS
# ============================================================

import os
import pandas as pd

RESULTS_FILE = os.path.join(
    OUTPUT_DIR,
    "metrics",
    "all_encoder_results.csv"
)

PROGRESS_FILE = os.path.join(
    OUTPUT_DIR,
    "metrics",
    "all_results_progress.csv"
)

# ------------------------------------------------------------
# LOAD PREVIOUS RESULTS
# ------------------------------------------------------------

existing_frames = []

if os.path.exists(PROGRESS_FILE):

    progress_df = pd.read_csv(
        PROGRESS_FILE
    )

    if len(progress_df) > 0:
        existing_frames.append(progress_df)

        print(
            "Loaded progress:",
            len(progress_df),
            "completed runs"
        )

if os.path.exists(RESULTS_FILE):

    final_df = pd.read_csv(
        RESULTS_FILE
    )

    if len(final_df) > 0:
        existing_frames.append(final_df)

        print(
            "Loaded final results:",
            len(final_df),
            "completed runs"
        )


if existing_frames:

    previous_results = pd.concat(
        existing_frames,
        ignore_index=True
    )

    previous_results = (
        previous_results
        .drop_duplicates(
            subset=[
                "task",
                "model",
                "seed"
            ],
            keep="last"
        )
        .reset_index(drop=True)
    )

else:

    previous_results = pd.DataFrame()


# ------------------------------------------------------------
# SHOW CURRENT PROGRESS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CURRENT EXPERIMENT PROGRESS")
print("=" * 80)

print(
    "Completed:",
    len(previous_results),
    "/ 30"
)

display(
    previous_results[
        [
            "task",
            "model",
            "seed",
            "Accuracy",
            "Macro_F1",
            "Weighted_F1"
        ]
    ].sort_values(
        [
            "task",
            "model",
            "seed"
        ]
    )
)


# ------------------------------------------------------------
# COMPLETED KEYS
# ------------------------------------------------------------

completed_keys = set()

for _, row in previous_results.iterrows():

    completed_keys.add(
        (
            str(row["task"]),
            str(row["model"]),
            int(row["seed"])
        )
    )


print("\nCompleted experiments:")

for key in sorted(completed_keys):
    print("SKIP:", key)


print(
    "\nRemaining:",
    30 - len(completed_keys)
)


# ------------------------------------------------------------
# PREVIOUS RESULTS
# ------------------------------------------------------------

all_results = previous_results.to_dict(
    orient="records"
)


# ============================================================
# RESUME TRAINING
# ============================================================

for task_name, task_info in TASKS.items():

    class_names = task_info["classes"]

    for seed in SEEDS:

        base = (
            f"{task_name}_seed{seed}"
        )

        train_original = pd.read_csv(
            os.path.join(
                OUTPUT_DIR,
                "splits",
                f"{base}_train_original.csv"
            )
        )

        train_os = pd.read_csv(
            os.path.join(
                OUTPUT_DIR,
                "splits",
                f"{base}_train_oversampled.csv"
            )
        )

        val_df = pd.read_csv(
            os.path.join(
                OUTPUT_DIR,
                "splits",
                f"{base}_validation.csv"
            )
        )

        test_df = pd.read_csv(
            os.path.join(
                OUTPUT_DIR,
                "splits",
                f"{base}_test.csv"
            )
        )


        for model_key in MODELS:

            current_key = (
                task_name,
                model_key,
                seed
            )


            # ------------------------------------------------
            # SKIP ALREADY COMPLETED
            # ------------------------------------------------

            if current_key in completed_keys:

                print(
                    f"\nSKIPPING COMPLETED: "
                    f"{task_name} | "
                    f"{model_key} | "
                    f"seed={seed}"
                )

                continue


            # ------------------------------------------------
            # TRAIN MISSING EXPERIMENT
            # ------------------------------------------------

            print("\n" + "=" * 80)

            print(
                f"RUNNING: "
                f"{task_name} | "
                f"{model_key} | "
                f"seed={seed}"
            )

            print("=" * 80)


            result = run_one_experiment(
                task_name=task_name,
                class_names=class_names,
                train_df=train_os,
                val_df=val_df,
                test_df=test_df,
                model_key=model_key,
                seed=seed,
            )


            result["train_original_n"] = (
                len(train_original)
            )

            result["train_oversampled_n"] = (
                len(train_os)
            )


            # ------------------------------------------------
            # SAVE RESULT IMMEDIATELY
            # ------------------------------------------------

            all_results.append(
                result
            )

            current_results = pd.DataFrame(
                all_results
            )

            current_results = (
                current_results
                .drop_duplicates(
                    subset=[
                        "task",
                        "model",
                        "seed"
                    ],
                    keep="last"
                )
            )


            current_results.to_csv(
                PROGRESS_FILE,
                index=False
            )

            current_results.to_csv(
                RESULTS_FILE,
                index=False
            )


            completed_keys.add(
                current_key
            )


            print(
                "\nCOMPLETED:",
                current_key
            )

            print(
                "Macro-F1:",
                result["Macro_F1"]
            )


            clear_gpu()


# ============================================================
# FINAL RESULTS
# ============================================================

results_df = pd.DataFrame(
    all_results
)

results_df = (
    results_df
    .drop_duplicates(
        subset=[
            "task",
            "model",
            "seed"
        ],
        keep="last"
    )
    .reset_index(drop=True)
)


results_df.to_csv(
    RESULTS_FILE,
    index=False
)


print("\n" + "=" * 80)
print("ALL RESUMED EXPERIMENTS FINISHED")
print("=" * 80)

print(
    "Completed:",
    len(results_df),
    "/ 30"
)

display(
    results_df[
        [
            "task",
            "model",
            "seed",
            "Accuracy",
            "Macro_F1",
            "Weighted_F1"
        ]
    ].sort_values(
        [
            "task",
            "model",
            "seed"
        ]
    )
)

Loaded progress: 10 completed runs

CURRENT EXPERIMENT PROGRESS
Completed: 10 / 30


,task,model,seed,Accuracy,Macro_F1,Weighted_F1
0,binary,BERT,42,0.993333,0.992608,0.993321
5,binary,BERT,123,0.994000,0.993356,0.993993
2,binary,DistilBERT,42,0.994000,0.993350,0.993990
7,binary,DistilBERT,123,0.995333,0.994832,0.995328
4,binary,MiniLM,42,0.991333,0.990385,0.991315
9,binary,MiniLM,123,0.992667,0.991880,0.992658
1,binary,RoBERTa,42,0.988000,0.986694,0.987978
6,binary,RoBERTa,123,0.991333,0.990412,0.991327
3,binary,TinyBERT,42,0.978667,0.976344,0.978627
8,binary,TinyBERT,123,0.982667,0.980833,0.982659



Completed experiments:
SKIP: ('binary', 'BERT', 42)
SKIP: ('binary', 'BERT', 123)
SKIP: ('binary', 'DistilBERT', 42)
SKIP: ('binary', 'DistilBERT', 123)
SKIP: ('binary', 'MiniLM', 42)
SKIP: ('binary', 'MiniLM', 123)
SKIP: ('binary', 'RoBERTa', 42)
SKIP: ('binary', 'RoBERTa', 123)
SKIP: ('binary', 'TinyBERT', 42)
SKIP: ('binary', 'TinyBERT', 123)

Remaining: 20

SKIPPING COMPLETED: binary | BERT | seed=42

SKIPPING COMPLETED: binary | RoBERTa | seed=42

SKIPPING COMPLETED: binary | DistilBERT | seed=42

SKIPPING COMPLETED: binary | TinyBERT | seed=42

SKIPPING COMPLETED: binary | MiniLM | seed=42

SKIPPING COMPLETED: binary | BERT | seed=123

SKIPPING COMPLETED: binary | RoBERTa | seed=123

SKIPPING COMPLETED: binary | DistilBERT | seed=123

SKIPPING COMPLETED: binary | TinyBERT | seed=123

SKIPPING COMPLETED: binary | MiniLM | seed=123

RUNNING: binary | BERT | seed=2024

TASK=binary | MODEL=BERT | SEED=2024
Attention=standard attention | Display=Without FA


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

... (59 outputs left out)

Epoch 1/3 | Loss: 1.20714 | Val Macro-F1: 0.74325
Epoch 2/3 | Loss: 0.57458 | Val Macro-F1: 0.81524
Epoch 3/3 | Loss: 0.40711 | Val Macro-F1: 0.82491

COMPLETED: ('five_class', 'MiniLM', 42)
Macro-F1: 0.8285141264424002

RUNNING: five_class | BERT | seed=123

TASK=five_class | MODEL=BERT | SEED=123
Attention=standard attention | Display=Without FA


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 | Loss: 0.88524 | Val Macro-F1: 0.78555
Epoch 2/3 | Loss: 0.25508 | Val Macro-F1: 0.81148


In [ ]:
# ============================================================
# 18. RUN ALL 30 ENCODER EXPERIMENTS
# ============================================================

all_results = []

for task_name, task_info in TASKS.items():

    class_names = task_info["classes"]

    for seed in SEEDS:

        base = f"{task_name}_seed{seed}"

        train_original = pd.read_csv(
            os.path.join(
                OUTPUT_DIR,
                "splits",
                f"{base}_train_original.csv"
            )
        )

        train_os = pd.read_csv(
            os.path.join(
                OUTPUT_DIR,
                "splits",
                f"{base}_train_oversampled.csv"
            )
        )

        val_df = pd.read_csv(
            os.path.join(
                OUTPUT_DIR,
                "splits",
                f"{base}_validation.csv"
            )
        )

        test_df = pd.read_csv(
            os.path.join(
                OUTPUT_DIR,
                "splits",
                f"{base}_test.csv"
            )
        )

        for model_key in MODELS:

            result = run_one_experiment(
                task_name=task_name,
                class_names=class_names,
                train_df=train_os,
                val_df=val_df,
                test_df=test_df,
                model_key=model_key,
                seed=seed,
            )

            result["train_original_n"] = len(
                train_original
            )

            result["train_oversampled_n"] = len(
                train_os
            )

            all_results.append(result)

            progress = pd.DataFrame(
                all_results
            )

            progress.to_csv(
                os.path.join(
                    OUTPUT_DIR,
                    "metrics",
                    "all_results_progress.csv"
                ),
                index=False
            )

            print(
                f"COMPLETED: {task_name} | "
                f"{model_key} | seed={seed}"
            )

            clear_gpu()

results_df = pd.DataFrame(
    all_results
)

results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "metrics",
        "all_encoder_results.csv"
    ),
    index=False
)

print("\nTotal completed runs:", len(results_df))
display(results_df)

In [ ]:
# ============================================================
# 19. DISPLAY-ONLY "WITH FA"
# ============================================================

# IMPORTANT:
# This does NOT execute FlashAttention.
# It creates a manuscript/display copy only.

with_fa_display = results_df.copy()

with_fa_display["display_condition"] = (
    DISPLAY_WITH_FA
)

with_fa_display["display_note"] = (
    "Display-only label. Actual attention is standard attention."
)

with_fa_display.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "metrics",
        "with_FA_DISPLAY_ONLY.csv"
    ),
    index=False
)

combined_display = pd.concat(
    [
        results_df,
        with_fa_display
    ],
    ignore_index=True
)

combined_display.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "metrics",
        "combined_display_results.csv"
    ),
    index=False
)

print(
    "With FA table created as DISPLAY ONLY. "
    "No FlashAttention kernel was executed."
)

In [ ]:
# ============================================================
# 20. MEAN / SD / 95% CI
# ============================================================

REPORT_METRICS = [
    "Accuracy",
    "Macro_Precision",
    "Macro_Recall",
    "Macro_F1",
    "Weighted_Precision",
    "Weighted_Recall",
    "Weighted_F1",
    "training_time_s",
    "latency_mean_ms",
    "latency_p95_ms",
    "peak_training_allocated_mb",
    "peak_inference_allocated_mb",
]

def mean_sd_ci(values):

    x = pd.to_numeric(
        pd.Series(values),
        errors="coerce"
    ).dropna().to_numpy()

    if len(x) == 0:
        return np.nan, np.nan, np.nan, np.nan

    mean = float(np.mean(x))

    sd = (
        float(np.std(x, ddof=1))
        if len(x) > 1
        else 0.0
    )

    if len(x) > 1:

        critical = student_t.ppf(
            0.975,
            len(x) - 1
        )

        margin = (
            critical * sd / np.sqrt(len(x))
        )

        return (
            mean,
            sd,
            mean - margin,
            mean + margin
        )

    return mean, sd, np.nan, np.nan


aggregate_rows = []

for (task, model), group in results_df.groupby(
    ["task", "model"]
):

    row = {
        "task": task,
        "model": model,
        "n_seeds": len(group),
        "display_condition": DISPLAY_WITHOUT_FA,
        "actual_attention":
            ACTUAL_ATTENTION_MECHANISM,
    }

    for metric in REPORT_METRICS:

        mean, sd, low, high = mean_sd_ci(
            group[metric]
        )

        row[f"{metric}_mean"] = mean
        row[f"{metric}_sd"] = sd
        row[f"{metric}_ci95_low"] = low
        row[f"{metric}_ci95_high"] = high

    aggregate_rows.append(row)

aggregate_df = pd.DataFrame(
    aggregate_rows
)

aggregate_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "metrics",
        "aggregate_mean_sd_ci95.csv"
    ),
    index=False
)

display(
    aggregate_df[
        [
            "task",
            "model",
            "n_seeds",
            "Accuracy_mean",
            "Macro_F1_mean",
            "Macro_F1_sd",
            "Macro_F1_ci95_low",
            "Macro_F1_ci95_high",
        ]
    ]
)

In [ ]:
# ============================================================
# 21. PAIRED BOOTSTRAP + MCNEMAR
# ============================================================

def load_prediction(task, model, seed):

    path = os.path.join(
        OUTPUT_DIR,
        "predictions",
        f"{task}_{model}_seed{seed}_WITHOUT_FA.csv"
    )

    if not os.path.exists(path):
        return None

    return pd.read_csv(path)


def paired_bootstrap_macro_f1(
    y_true,
    pred_a,
    pred_b,
    n_boot=BOOTSTRAP_ITERATIONS,
    seed=42,
):

    rng = np.random.default_rng(seed)

    y_true = np.asarray(y_true)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)

    observed = (
        f1_score(
            y_true,
            pred_b,
            average="macro",
            zero_division=0
        )
        -
        f1_score(
            y_true,
            pred_a,
            average="macro",
            zero_division=0
        )
    )

    n = len(y_true)

    deltas = np.empty(n_boot)

    for i in range(n_boot):

        indices = rng.integers(
            0,
            n,
            size=n
        )

        deltas[i] = (
            f1_score(
                y_true[indices],
                pred_b[indices],
                average="macro",
                zero_division=0
            )
            -
            f1_score(
                y_true[indices],
                pred_a[indices],
                average="macro",
                zero_division=0
            )
        )

    low, high = np.percentile(
        deltas,
        [2.5, 97.5]
    )

    p = 2 * min(
        np.mean(deltas <= 0),
        np.mean(deltas >= 0)
    )

    return {
        "delta_macro_f1":
            float(observed),
        "bootstrap_ci95_low":
            float(low),
        "bootstrap_ci95_high":
            float(high),
        "bootstrap_p_value":
            float(min(1.0, p)),
    }


def paired_mcnemar(
    y_true,
    pred_a,
    pred_b
):

    y_true = np.asarray(y_true)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)

    a_correct = pred_a == y_true
    b_correct = pred_b == y_true

    table = np.array([
        [
            np.sum(
                a_correct & b_correct
            ),
            np.sum(
                a_correct & ~b_correct
            ),
        ],
        [
            np.sum(
                ~a_correct & b_correct
            ),
            np.sum(
                ~a_correct & ~b_correct
            ),
        ],
    ])

    result = mcnemar(
        table,
        exact=True,
        correction=False
    )

    return {
        "a_correct_b_correct":
            int(table[0, 0]),
        "a_correct_b_wrong":
            int(table[0, 1]),
        "a_wrong_b_correct":
            int(table[1, 0]),
        "a_wrong_b_wrong":
            int(table[1, 1]),
        "mcnemar_statistic":
            float(result.statistic),
        "mcnemar_p_value":
            float(result.pvalue),
    }


stat_rows = []

for task_name in TASKS:

    for seed in SEEDS:

        prediction_cache = {}

        for model in MODELS:

            prediction = load_prediction(
                task_name,
                model,
                seed
            )

            if prediction is not None:
                prediction_cache[model] = prediction

        model_names = list(
            prediction_cache.keys()
        )

        for i, model_a in enumerate(
            model_names
        ):

            for model_b in model_names[i + 1:]:

                a = prediction_cache[model_a]
                b = prediction_cache[model_b]

                merged = a.merge(
                    b,
                    on="original_row_id",
                    suffixes=("_A", "_B")
                )

                # True labels must agree.
                merged = merged[
                    merged["TrueLabel_A"]
                    ==
                    merged["TrueLabel_B"]
                ]

                if len(merged) == 0:
                    continue

                bootstrap = (
                    paired_bootstrap_macro_f1(
                        merged["TrueLabel_A"].to_numpy(),
                        merged["Prediction_A"].to_numpy(),
                        merged["Prediction_B"].to_numpy(),
                        seed=seed
                    )
                )

                mcnemar_result = paired_mcnemar(
                    merged["TrueLabel_A"].to_numpy(),
                    merged["Prediction_A"].to_numpy(),
                    merged["Prediction_B"].to_numpy()
                )

                stat_rows.append({
                    "task": task_name,
                    "seed": seed,
                    "model_a": model_a,
                    "model_b": model_b,
                    "n_matched": len(merged),
                    **bootstrap,
                    **mcnemar_result
                })

statistics_df = pd.DataFrame(
    stat_rows
)

statistics_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "statistics",
        "paired_bootstrap_mcnemar.csv"
    ),
    index=False
)

display(
    statistics_df.head(20)
)

In [ ]:
# ============================================================
# 22. QUALITATIVE ERROR ANALYSIS
# ============================================================

error_frames = []

for task_name in TASKS:

    for model in MODELS:

        for seed in SEEDS:

            prediction = load_prediction(
                task_name,
                model,
                seed
            )

            if prediction is None:
                continue

            errors = prediction[
                prediction["TrueLabel"]
                !=
                prediction["Prediction"]
            ].copy()

            if len(errors) == 0:
                continue

            errors["task"] = task_name
            errors["model"] = model
            errors["seed"] = seed

            error_frames.append(errors)

if error_frames:

    all_errors = pd.concat(
        error_frames,
        ignore_index=True
    )

    all_errors.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "qualitative",
            "all_error_cases.csv"
        ),
        index=False
    )

    print(
        "Total error records saved:",
        len(all_errors)
    )

else:
    print("No error records available.")

# 23. Optional Mistral baseline

This section is intentionally disabled.

Set:

```python
RUN_MISTRAL = True
```

only if you want the optional zero-shot Mistral comparison.

The encoder experiments should be completed first.

In [ ]:
# ============================================================
# 23. OPTIONAL MISTRAL ZERO-SHOT BASELINE
# ============================================================

def run_mistral_zero_shot(
    task_name,
    test_csv,
    output_csv
):

    from transformers import (
        AutoTokenizer,
        AutoModelForCausalLM,
        BitsAndBytesConfig,
    )

    test_df = pd.read_csv(
        test_csv
    )

    labels = (
        BINARY_CLASSES
        if task_name == "binary"
        else FIVE_CLASSES
    )

    tokenizer = AutoTokenizer.from_pretrained(
        MISTRAL_MODEL
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    quantization_config = None

    if MISTRAL_4BIT and torch.cuda.is_available():

        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )

    model = AutoModelForCausalLM.from_pretrained(
        MISTRAL_MODEL,
        device_map="auto",
        torch_dtype=(
            torch.float16
            if torch.cuda.is_available()
            else torch.float32
        ),
        quantization_config=quantization_config,
    )

    model.eval()

    rows = []

    for _, row in test_df.iterrows():

        prompt = (
            "Classify this mobile app review into exactly "
            "one of these labels:\n"
            f"{', '.join(labels)}\n\n"
            "Return ONLY the label.\n\n"
            f"Review: {row['comment_Text']}"
        )

        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        if hasattr(
            tokenizer,
            "apply_chat_template"
        ):

            rendered = (
                tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True
                )
            )

        else:
            rendered = prompt

        inputs = tokenizer(
            rendered,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LEN
        )

        if torch.cuda.is_available():
            inputs = {
                k: v.to(model.device)
                for k, v in inputs.items()
            }

        start = time.perf_counter()

        with torch.no_grad():

            generated = model.generate(
                **inputs,
                max_new_tokens=MISTRAL_MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed = (
            time.perf_counter() - start
        )

        prompt_length = (
            inputs["input_ids"].shape[1]
        )

        generated_tokens = (
            generated[0, prompt_length:]
        )

        raw_output = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()

        normalized = re.sub(
            r"[\s\.,:;!?_\-]+",
            " ",
            raw_output.lower()
        ).strip()

        prediction = None

        for label in labels:

            if normalized == label.lower():
                prediction = label
                break

        if prediction is None:

            matches = [
                label
                for label in labels
                if label.lower()
                in normalized
            ]

            if len(matches) == 1:
                prediction = matches[0]

        rows.append({
            "original_row_id":
                row["original_row_id"],
            "TrueLabel":
                row["label_name"],
            "MistralPrediction":
                prediction,
            "RawOutput":
                raw_output,
            "valid_prediction":
                prediction is not None,
            "latency_s":
                elapsed,
        })

    result = pd.DataFrame(rows)

    result.to_csv(
        output_csv,
        index=False
    )

    return result


if RUN_MISTRAL:

    for task_name in TASKS:

        test_csv = os.path.join(
            OUTPUT_DIR,
            "splits",
            f"{task_name}_seed42_test.csv"
        )

        output_csv = os.path.join(
            OUTPUT_DIR,
            "mistral",
            f"Mistral_{task_name}_seed42.csv"
        )

        run_mistral_zero_shot(
            task_name,
            test_csv,
            output_csv
        )

else:

    print(
        "Mistral baseline disabled."
    )

In [ ]:
# ============================================================
# 24. FATIMA ET AL. COMPARISON TABLE
# ============================================================
# These are the supplied comparison values from the earlier pipeline.
# They are kept separate from newly measured experimental results.

fatima_data = [
    {
        "model": "LSTM",
        "accuracy_percent": 91,
        "humor_f1": .86,
        "flattery_f1": .93,
        "self_deprecation_f1": .90,
        "irony_f1": .90,
        "passive_aggression_f1": .93,
    },
    {
        "model": "BiLSTM",
        "accuracy_percent": 92,
        "humor_f1": .88,
        "flattery_f1": .93,
        "self_deprecation_f1": .92,
        "irony_f1": .91,
        "passive_aggression_f1": .95,
    },
    {
        "model": "CNN",
        "accuracy_percent": 90,
        "humor_f1": .94,
        "flattery_f1": .96,
        "self_deprecation_f1": .93,
        "irony_f1": .98,
        "passive_aggression_f1": .91,
    },
    {
        "model": "GRU",
        "accuracy_percent": 91,
        "humor_f1": .95,
        "flattery_f1": .91,
        "self_deprecation_f1": .92,
        "irony_f1": .94,
        "passive_aggression_f1": .90,
    },
    {
        "model": "BiGRU",
        "accuracy_percent": 91,
        "humor_f1": .98,
        "flattery_f1": .95,
        "self_deprecation_f1": .94,
        "irony_f1": .96,
        "passive_aggression_f1": .92,
    },
    {
        "model": "RNN",
        "accuracy_percent": 75,
        "humor_f1": .87,
        "flattery_f1": .95,
        "self_deprecation_f1": .90,
        "irony_f1": .94,
        "passive_aggression_f1": .87,
    },
    {
        "model": "BiRNN",
        "accuracy_percent": 89,
        "humor_f1": .93,
        "flattery_f1": .96,
        "self_deprecation_f1": .93,
        "irony_f1": .98,
        "passive_aggression_f1": .90,
    },
]

fatima_df = pd.DataFrame(
    fatima_data
)

fatima_df["derived_macro_f1"] = (
    fatima_df[
        [
            "humor_f1",
            "flattery_f1",
            "self_deprecation_f1",
            "irony_f1",
            "passive_aggression_f1",
        ]
    ].mean(axis=1)
)

fatima_df["result_type"] = (
    "Supplied classwise F1; Macro-F1 derived"
)

fatima_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "fatima",
        "Fatima_comparison.csv"
    ),
    index=False
)

display(fatima_df)

In [ ]:
# ============================================================
# 25. MANUSCRIPT-READY METHODS TEXT
# ============================================================

methods_text = f"""
Experimental protocol.

Two datasets were used for separate classification tasks. The binary task
used the annotated_comments_binary_1(1).csv dataset, where "No" in
Have_Sarcasm was mapped to Non-Sarcastic and "Yes" was mapped to Sarcastic.
The five-class task used the Sarcasm_multiclassification_1(1).csv dataset
and classified Humor, Irony, Flattery, Self-Deprecation, and
Passive-Aggression.

For each task, the data were independently stratified into 70% training,
15% validation, and 15% test partitions using three random seeds
(42, 123, and 2024). RandomOverSampler was applied only to the training
partition after splitting. Validation and test partitions were never
oversampled.

Five pretrained Transformer architectures were evaluated: BERT, RoBERTa,
DistilBERT, TinyBERT, and MiniLM. Evaluation included Accuracy, macro and
weighted Precision, Recall, and F1, per-class metrics, confusion matrices,
training time, inference latency, P95 latency, and GPU memory usage.
Results were aggregated using mean, standard deviation, and 95% confidence
intervals. Paired bootstrap and McNemar analyses were also performed for
model comparisons.

For manuscript/display compatibility, the labels "{DISPLAY_WITHOUT_FA}"
and "{DISPLAY_WITH_FA}" may be used. Both conditions in the current
implementation use {ACTUAL_ATTENTION_MECHANISM}; no FlashAttention CUDA
kernel is executed.
"""

with open(
    os.path.join(
        OUTPUT_DIR,
        "manuscript",
        "methods_revision.txt"
    ),
    "w"
) as f:
    f.write(methods_text.strip())

print(methods_text)

In [ ]:
# ============================================================
# 26. FINAL AUDIT + RESULTS SUMMARY
# ============================================================

final_audit = {
    "five_class_source":
        "Sarcasm_multiclassification_1(1).csv",

    "binary_source":
        "annotated_comments_binary_1(1).csv",

    "five_class_rows":
        len(five_class_df),

    "binary_rows":
        len(binary_valid),

    "five_class_counts":
        five_class_df["label_name"].value_counts().to_dict(),

    "binary_counts":
        binary_valid["label_name"].value_counts().to_dict(),

    "datasets_concatenated":
        False,

    "train_only_oversampling":
        True,

    "validation_oversampling":
        False,

    "test_oversampling":
        False,

    "models":
        list(MODELS.keys()),

    "seeds":
        SEEDS,

    "expected_encoder_runs":
        30,

    "actual_completed_runs":
        len(results_df),

    "actual_attention":
        ACTUAL_ATTENTION_MECHANISM,

    "flash_attention_used":
        False,

    "with_fa_is_display_only":
        True,

    "binary_one_class_error_fixed":
        True,
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "audit",
        "FINAL_AUDIT.json"
    ),
    "w"
) as f:
    json.dump(
        final_audit,
        f,
        indent=2,
        default=str
    )

print("=" * 78)
print("FINAL EXPERIMENT AUDIT")
print("=" * 78)
print(json.dumps(
    final_audit,
    indent=2,
    default=str
))

print("\nResults saved in:")
print(OUTPUT_DIR)

# 27. Important interpretation

### Binary task

Dataset B is used:

`annotated_comments_binary_1(1).csv`

```text
Non-Sarcastic    6539
Sarcastic        3460
```

This is a valid binary classification problem.

### Five-class task

Dataset A is used:

`Sarcasm_multiclassification_1(1).csv`

```text
Humor                 1709
Irony                  536
Flattery               513
Self-Deprecation       439
Passive-Aggression     263
```

This is a valid five-class classification problem.

### Do not concatenate the datasets

The two files are used as separate experimental datasets/tasks.

### Oversampling

RandomOverSampler is applied only to the training partition. This prevents validation/test distribution contamination.

### FlashAttention

The labels `With FA` and `Without FA` are display labels only. The actual implementation uses standard attention. Therefore, do not describe this experiment as a real FlashAttention-vs-standard-attention benchmark unless you separately implement and execute FlashAttention.